# Week 11 Quantile Training (Colab)

This notebook trains Week 11 quantile artifacts from `auction_snapshots.csv` using the same snapshot schema as Week 9.

Outputs:
- `models/quantile_models.pkl`
- `models/week11_quantile_metrics.json`
- `models/week11_quantile_metadata.json`

Use this after the Week 9 point-model pass. The Streamlit app can keep the Week 9 point estimate and read these Week 11 quantiles for q10, q50, q75, and q90.


In [1]:
# Install dependencies (Colab)
!pip -q install numpy pandas scikit-learn


In [2]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_pinball_loss, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from google.colab import files
except ImportError:
    files = None


Upload Kush's `auction_snapshots.csv` or run this notebook in a folder where that file already exists.


In [3]:
# Upload teammate snapshot CSV
# Expected file: auction_snapshots.csv
if files is None:
    INPUT_CSV = "auction_snapshots.csv"
    print("Running outside Colab; using local file:", Path(INPUT_CSV).resolve())
else:
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise RuntimeError("No file uploaded.")
    INPUT_CSV = next(iter(uploaded))
    print("Using uploaded file:", INPUT_CSV)


Saving auction_snapshots.csv to auction_snapshots (1).csv
Using uploaded file: auction_snapshots (1).csv


In [4]:
TARGET_COLUMN = "final_price"
GROUP_COLUMN = "auction_id"

REQUIRED_COLUMNS = [
    "auction_id",
    "item_name",
    "auction_type",
    "auction_days",
    "snapshot_pct",
    "snapshot_time_days",
    "estimated_bid_amount",
    "opening_bid",
    "final_price",
    "bid_count_so_far",
    "unique_bidders_so_far",
    "max_observed_bid_so_far",
    "leading_bidder_so_far",
    "leading_bidder_rate_so_far",
]

CATEGORICAL_COLUMNS = ["item_name", "auction_type", "snapshot_pct"]
NUMERIC_COLUMNS = [
    "auction_days",
    "snapshot_time_days",
    "opening_bid",
    "bid_count_so_far",
    "unique_bidders_so_far",
    "max_observed_bid_so_far",
    "leading_bidder_rate_so_far",
]
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS

QUANTILE_SPECS = {
    "q10": 0.10,
    "q50": 0.50,
    "q75": 0.75,
    "q90": 0.90,
}


In [5]:
def metric_dict(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {"mae": mae, "rmse": rmse}


def quantile_metric_dict(y_true, y_pred, alpha):
    out = metric_dict(y_true, y_pred)
    out["pinball_loss"] = float(mean_pinball_loss(y_true, y_pred, alpha=alpha))
    return out


def interval_metric_dict(y_true, pred_df, low_label="q10", high_label="q90"):
    within = (y_true >= pred_df[low_label]) & (y_true <= pred_df[high_label])
    width = pred_df[high_label] - pred_df[low_label]
    return {
        "coverage": float(within.mean()),
        "avg_interval_width": float(width.mean()),
    }


def load_and_validate(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing input CSV: {csv_path.resolve()}")

    df = pd.read_csv(csv_path)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    numeric_to_parse = [
        "auction_days",
        "snapshot_pct",
        "snapshot_time_days",
        "estimated_bid_amount",
        "opening_bid",
        "final_price",
        "bid_count_so_far",
        "unique_bidders_so_far",
        "max_observed_bid_so_far",
        "leading_bidder_rate_so_far",
    ]
    for col in numeric_to_parse:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["snapshot_pct"] = df["snapshot_pct"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "MISSING")
    for col in ["item_name", "auction_type", "leading_bidder_so_far"]:
        df[col] = df[col].astype("string").fillna("MISSING").astype(str)

    df[GROUP_COLUMN] = df[GROUP_COLUMN].astype("string").fillna("MISSING").astype(str)

    before = len(df)
    df = df.dropna(subset=[TARGET_COLUMN]).copy()
    dropped_target = before - len(df)
    if dropped_target:
        print(f"Dropped {dropped_target} rows with missing target ({TARGET_COLUMN}).")

    return df


def make_split(df, test_size=0.2, random_state=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(df, df[TARGET_COLUMN], groups=df[GROUP_COLUMN]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    overlap = set(train_df[GROUP_COLUMN]).intersection(set(test_df[GROUP_COLUMN]))
    if overlap:
        raise RuntimeError(f"Leakage guard failed: {len(overlap)} auction IDs overlap train/test.")

    return train_df, test_df


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                CATEGORICAL_COLUMNS,
            ),
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), NUMERIC_COLUMNS),
        ],
        remainder="drop",
    )


def train_quantile_model(
    train_df,
    alpha,
    random_state=42,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=5,
):
    preprocessor = make_preprocessor()
    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=alpha,
        random_state=random_state,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
    )
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    x_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET_COLUMN].to_numpy(dtype=float)
    pipeline.fit(x_train, y_train)
    return pipeline


def aggregate_raw_feature_importance(model_pipeline):
    pre = model_pipeline.named_steps["preprocessor"]
    model = model_pipeline.named_steps["model"]

    names = pre.get_feature_names_out()
    importances = model.feature_importances_
    agg = {name: 0.0 for name in FEATURE_COLUMNS}

    for encoded_name, imp in zip(names, importances):
        if encoded_name.startswith("num__"):
            raw = encoded_name.replace("num__", "", 1)
        elif encoded_name.startswith("cat__"):
            tail = encoded_name.replace("cat__", "", 1)
            raw = None
            for col in CATEGORICAL_COLUMNS:
                prefix = f"{col}_"
                if tail == col or tail.startswith(prefix):
                    raw = col
                    break
            if raw is None:
                raw = tail.split("_", 1)[0]
        else:
            raw = encoded_name
        agg[raw] = float(agg.get(raw, 0.0) + float(imp))

    sorted_items = sorted(agg.items(), key=lambda kv: kv[1], reverse=True)
    return {k: float(v) for k, v in sorted_items}


def enforce_quantile_order(pred_df, labels=None):
    labels = labels or list(QUANTILE_SPECS.keys())
    arr = pred_df[labels].to_numpy(dtype=float)
    arr = np.maximum.accumulate(arr, axis=1)
    out = pred_df.copy()
    out[labels] = arr
    return out


def ensure_serializable_metrics(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[str(k)] = ensure_serializable_metrics(v)
        elif isinstance(v, list):
            out[str(k)] = [ensure_serializable_metrics(x) if isinstance(x, dict) else x for x in v]
        elif isinstance(v, np.floating):
            out[str(k)] = float(v)
        elif isinstance(v, np.integer):
            out[str(k)] = int(v)
        else:
            out[str(k)] = v
    return out


In [6]:
# Train Week 11 quantile models
df = load_and_validate(INPUT_CSV)
train_df, test_df = make_split(df, test_size=0.2, random_state=42)

print("Rows total:", len(df))
print("Rows train:", len(train_df))
print("Rows test:", len(test_df))
print("Auctions train:", train_df[GROUP_COLUMN].nunique())
print("Auctions test:", test_df[GROUP_COLUMN].nunique())

overlap = set(train_df[GROUP_COLUMN]).intersection(set(test_df[GROUP_COLUMN]))
print("Train/Test auction overlap:", len(overlap))

y_test = test_df[TARGET_COLUMN].to_numpy(dtype=float)
quantile_models = {}

raw_pred_df = pd.DataFrame(
    {
        "y_true": y_test,
        "snapshot_pct": test_df["snapshot_pct"].astype(str).to_numpy(),
        "item_name": test_df["item_name"].astype(str).to_numpy(),
    }
)

for label, alpha in QUANTILE_SPECS.items():
    print(f"Training {label} (alpha={alpha:.2f}) ...")
    model = train_quantile_model(
        train_df,
        alpha=alpha,
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=5,
    )
    quantile_models[label] = model
    raw_pred_df[label] = model.predict(test_df[FEATURE_COLUMNS])

crossing_mask = (
    (raw_pred_df["q10"] > raw_pred_df["q50"])
    | (raw_pred_df["q50"] > raw_pred_df["q75"])
    | (raw_pred_df["q75"] > raw_pred_df["q90"])
)
crossing_rate = float(crossing_mask.mean())

pred_df = enforce_quantile_order(raw_pred_df)

overall_rows = []
for label, alpha in QUANTILE_SPECS.items():
    row = {"quantile": label, "alpha": alpha, **quantile_metric_dict(y_test, pred_df[label].to_numpy(dtype=float), alpha)}
    overall_rows.append(row)

overall_df = pd.DataFrame(overall_rows).sort_values("alpha")
print("Quantile crossing rate before enforcement:", round(crossing_rate, 4))
overall_df


Rows total: 4396
Rows train: 3514
Rows test: 882
Auctions train: 502
Auctions test: 126
Train/Test auction overlap: 0
Training q10 (alpha=0.10) ...
Training q50 (alpha=0.50) ...
Training q75 (alpha=0.75) ...
Training q90 (alpha=0.90) ...
Quantile crossing rate before enforcement: 0.1927


,quantile,alpha,mae,rmse,pinball_loss
0,q10,0.10,95.140912,276.957368,11.489406
1,q50,0.50,58.502480,157.999726,29.251240
2,q75,0.75,62.761415,151.237978,28.802982
3,q90,0.90,101.067445,229.861801,21.898758


In [7]:
# Interval diagnostics
interval_summary = interval_metric_dict(y_test, pred_df, low_label="q10", high_label="q90")
interval_summary

by_stage_rows = []
for stage, stage_frame in pred_df.groupby("snapshot_pct"):
    y_stage = stage_frame["y_true"].to_numpy(dtype=float)
    row = {
        "snapshot_pct": stage,
        **interval_metric_dict(y_stage, stage_frame, low_label="q10", high_label="q90"),
    }
    for label, alpha in QUANTILE_SPECS.items():
        row[f"{label}_pinball_loss"] = float(
            mean_pinball_loss(y_stage, stage_frame[label].to_numpy(dtype=float), alpha=alpha)
        )
    by_stage_rows.append(row)

by_stage_df = pd.DataFrame(by_stage_rows).sort_values("snapshot_pct")
by_stage_df


,snapshot_pct,coverage,avg_interval_width,q10_pinball_loss,q50_pinball_loss,q75_pinball_loss,q90_pinball_loss
0,0.25,0.785714,212.068073,17.634305,46.015946,42.190011,36.345899
1,0.50,0.753968,196.352523,15.263109,44.909028,44.456992,32.121514
2,0.75,0.769841,166.537880,10.641477,29.031563,28.652068,20.429807
3,0.85,0.753968,160.808848,10.085150,27.069657,28.111418,20.583031
4,0.90,0.753968,153.985076,10.385116,26.053860,27.664798,21.292051
5,0.95,0.761905,155.651905,9.322218,20.850282,22.298175,16.446537
6,1.00,0.682540,87.125945,7.094467,10.828343,8.247411,6.072469


In [8]:
# Save Week 11 quantile artifacts
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

numeric_medians = train_df[NUMERIC_COLUMNS].median().replace({np.nan: None}).to_dict()
numeric_medians = {k: (float(v) if v is not None and pd.notna(v) else None) for k, v in numeric_medians.items()}

feature_importance_by_quantile = {
    label: aggregate_raw_feature_importance(model)
    for label, model in quantile_models.items()
}

metrics = {
    "dataset": {
        "input_csv": str(Path(INPUT_CSV).resolve()),
        "rows_total": int(len(df)),
        "rows_train": int(len(train_df)),
        "rows_test": int(len(test_df)),
        "auctions_total": int(df[GROUP_COLUMN].nunique()),
        "auctions_train": int(train_df[GROUP_COLUMN].nunique()),
        "auctions_test": int(test_df[GROUP_COLUMN].nunique()),
        "snapshot_pct_values": sorted([str(v) for v in df["snapshot_pct"].dropna().unique()]),
    },
    "leakage_check": {
        "train_test_auction_overlap_count": int(len(set(train_df[GROUP_COLUMN]).intersection(set(test_df[GROUP_COLUMN]))))
    },
    "overall_quantiles": {
        label: quantile_metric_dict(y_test, pred_df[label].to_numpy(dtype=float), alpha)
        for label, alpha in QUANTILE_SPECS.items()
    },
    "intervals": {
        "q10_q90": interval_summary,
        "crossing_rate_before_enforcement": crossing_rate,
    },
    "by_snapshot_pct": {
        row["snapshot_pct"]: {
            "coverage": row["coverage"],
            "avg_interval_width": row["avg_interval_width"],
            "q10_pinball_loss": row["q10_pinball_loss"],
            "q50_pinball_loss": row["q50_pinball_loss"],
            "q75_pinball_loss": row["q75_pinball_loss"],
            "q90_pinball_loss": row["q90_pinball_loss"],
        }
        for row in by_stage_rows
    },
}

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(Path(INPUT_CSV).resolve()),
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "target_column": TARGET_COLUMN,
    "group_column": GROUP_COLUMN,
    "snapshot_stage_column": "snapshot_pct",
    "quantile_levels": QUANTILE_SPECS,
    "numeric_feature_medians": numeric_medians,
    "feature_importance_raw_by_quantile": feature_importance_by_quantile,
    "postprocess": {
        "enforce_quantile_order": True,
        "ordered_labels": list(QUANTILE_SPECS.keys()),
    },
    "limitations": [
        "Built from historical sold-auction snapshots only.",
        "No seller reputation, item condition text, images, or calendar-time context.",
        "Proxy-bidding behavior may make bidding dynamics appear non-monotonic.",
        "Quantile estimates are empirical and may still cross without post-processing on new inputs.",
    ],
}

model_artifact = {
    "models": quantile_models,
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "target_column": TARGET_COLUMN,
    "quantile_levels": QUANTILE_SPECS,
    "postprocess": {
        "enforce_quantile_order": True,
        "ordered_labels": list(QUANTILE_SPECS.keys()),
    },
}

with open(models_dir / "quantile_models.pkl", "wb") as f:
    pickle.dump(model_artifact, f)

with open(models_dir / "week11_quantile_metrics.json", "w", encoding="utf-8") as f:
    json.dump(ensure_serializable_metrics(metrics), f, indent=2)

with open(models_dir / "week11_quantile_metadata.json", "w", encoding="utf-8") as f:
    json.dump(ensure_serializable_metrics(metadata), f, indent=2)

print("Saved:")
print(" -", (models_dir / "quantile_models.pkl").resolve())
print(" -", (models_dir / "week11_quantile_metrics.json").resolve())
print(" -", (models_dir / "week11_quantile_metadata.json").resolve())


Saved:
 - /content/models/quantile_models.pkl
 - /content/models/week11_quantile_metrics.json
 - /content/models/week11_quantile_metadata.json


In [9]:
# Minimal Week 11 quantile prediction interface (in-notebook)
def predict_quantiles(
    snapshot_record,
    model_path="models/quantile_models.pkl",
    metadata_path="models/week11_quantile_metadata.json",
):
    with open(model_path, "rb") as f:
        model_artifact = pickle.load(f)
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    feature_columns = model_artifact["feature_columns"]
    labels = model_artifact.get("postprocess", {}).get("ordered_labels", list(model_artifact["quantile_levels"].keys()))
    models = model_artifact["models"]

    row = {feature: snapshot_record.get(feature) for feature in feature_columns}
    frame = pd.DataFrame([row])
    raw = np.array([float(models[label].predict(frame)[0]) for label in labels], dtype=float)
    ordered = np.maximum.accumulate(raw)

    out = {label: round(float(value), 4) for label, value in zip(labels, ordered)}
    out["limitations"] = metadata.get("limitations", [])
    return out


sample = test_df.iloc[0]
sample_input = {c: sample[c] for c in FEATURE_COLUMNS}
predict_quantiles(sample_input)


{'q10': 194.6707,
 'q50': 282.0277,
 'q75': 456.6393,
 'q90': 480.9718,
 'limitations': ['Built from historical sold-auction snapshots only.',
  'No seller reputation, item condition text, images, or calendar-time context.',
  'Proxy-bidding behavior may make bidding dynamics appear non-monotonic.',
  'Quantile estimates are empirical and may still cross without post-processing on new inputs.']}

In [10]:
# Optional: download artifacts from Colab
if files is None:
    print("files.download is only available in Google Colab.")
else:
    files.download("models/quantile_models.pkl")
    files.download("models/week11_quantile_metrics.json")
    files.download("models/week11_quantile_metadata.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>